# Verifier Agent

**Pipeline**

Retrieved Products → Review Summaries → Ranking Agent → Verifier Agent → Final Recommendation

This notebook is aligned with the updated `verifier_agent.py` implementation:
- no currency field is passed to the verifier;
- budget is numeric;
- review evidence is preserved for feature verification;
- ranking and verification outputs are displayed clearly.


In [16]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from Agents.retrieval_agent import RetrievalAgent, ProductQuery
from Agents.review_agent import ReviewAgent
from Agents.ranking_agent import RankingAgent
from Agents.verifier_agent import VerifierAgent


In [17]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [18]:
vector_db = FAISS.load_local(
    "../Data/Cleaned/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

reviews_df = pd.read_parquet(
    "../Data/Cleaned/reviews_sample.parquet"
)

print("FAISS index and review data loaded successfully.")
print("Review rows:", len(reviews_df))


FAISS index and review data loaded successfully.
Review rows: 26031


In [19]:
retrieval_agent = RetrievalAgent(vector_db)

review_agent = ReviewAgent(
    llm,
    reviews_df
)

ranking_agent = RankingAgent(llm)

verifier_agent = VerifierAgent(llm)


In [20]:
query = ProductQuery(
    product_type="phone",
    brand="Motorola",
    budget=50000,
    features=[
        "good battery life",
        "good sound quality"
    ]
)

print("Query:", query)


Query: product_type='phone' brand='Motorola' budget=50000.0 features=['good battery life', 'good sound quality']


In [21]:
retrieved_products = retrieval_agent.retrieve(query)

print(
    f"Retrieved {len(retrieved_products)} products"
)

if not retrieved_products:
    raise ValueError(
        "No products were retrieved."
    )


Retrieved 4 products


In [22]:
def safe_float(value, default=0.0):
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def safe_int(value, default=0):
    try:
        if value is None or pd.isna(value):
            return default
        return int(float(value))
    except (TypeError, ValueError):
        return default


In [23]:
products = []

for doc, retrieval_score in retrieved_products:

    parent_asin = str(
        doc.metadata.get(
            "parent_asin",
            ""
        )
    )

    try:
        review = review_agent.summarize_reviews(
            parent_asin
        )

        if hasattr(review, "model_dump"):
            review = review.model_dump()

        elif not isinstance(review, dict):
            review = {}

    except Exception as e:

        print(
            f"Review summarisation failed for "
            f"{parent_asin}: {e}"
        )

        review = {}

    product = {
        "parent_asin": parent_asin,

        "title": str(
            doc.metadata.get(
                "title",
                ""
            )
        ),

        "price": doc.metadata.get(
            "price"
        ),

        "retrieval_score": safe_float(
            doc.metadata.get(
                "retrieval_score",
                retrieval_score
            )
        ),

        "average_rating": safe_float(
            doc.metadata.get(
                "average_rating",
                0
            )
        ),

        "rating_number": safe_int(
            doc.metadata.get(
                "rating_number",
                0
            )
        ),

        "review_count": safe_int(
            review.get(
                "review_count",
                0
            )
        ),

        "overall_sentiment": review.get(
            "overall_sentiment",
            "unknown"
        ),

        "pros": review.get(
            "pros",
            []
        ) or [],

        "cons": review.get(
            "cons",
            []
        ) or [],

        "recommended_for": review.get(
            "recommended_for",
            ""
        ) or "",

        "avoid_if": review.get(
            "avoid_if",
            ""
        ) or "",

        "review_summary": review.get(
            "summary",
            review.get(
                "review_summary",
                ""
            )
        ) or "",

        # Additional metadata used by build_feature_evidence()
        "description": doc.metadata.get(
            "description",
            ""
        ) or "",

        "features": doc.metadata.get(
            "features",
            ""
        ) or "",

        "highlights": doc.metadata.get(
            "highlights",
            ""
        ) or "",

        "product_text": doc.metadata.get(
            "product_text",
            doc.page_content
        ) or "",

        "product_description": doc.metadata.get(
            "product_description",
            ""
        ) or "",

        "categories": doc.metadata.get(
            "categories",
            []
        ) or []
    }

    products.append(product)


if not products:
    raise ValueError(
        "No products are available for ranking."
    )

print(
    f"Prepared {len(products)} products "
    "with metadata and review evidence."
)


Prepared 4 products with metadata and review evidence.


In [24]:
products_df = pd.DataFrame(products)

display_columns = [
    "parent_asin",
    "title",
    "price",
    "retrieval_score",
    "average_rating",
    "rating_number",
    "review_count",
    "overall_sentiment",
    "review_summary"
]

display(
    products_df[
        [
            column
            for column in display_columns
            if column in products_df.columns
        ]
    ]
)


,parent_asin,title,price,retrieval_score,average_rating,rating_number,review_count,overall_sentiment,review_summary
0,B09KZ9P939,eMorevalue Earbuds Noise Cancelling Magnetic E...,10.68,0.451585,3.8,77,0,Unknown,No customer reviews available.
1,B07D5MRRFG,"Cellet Retractable Stereo in-Ear Headphone, Ea...",NaN,0.448896,4.5,4,0,Unknown,No customer reviews available.
2,B094XGTNGN,"Moto G7+ Plus (64GB, 4GB) 6.2"" FHD+ Max Vision...",NaN,0.422461,3.3,14,0,Unknown,No customer reviews available.
3,B00IMSG0V0,Premium Faux Leather Vertical Swivel Belt Clip...,14.99,0.373908,1.9,2,0,Unknown,No customer reviews available.


In [25]:
ranking = ranking_agent.rank_products(
    query,
    products
)

ranked_products = getattr(
    ranking,
    "products",
    []
) or []

if not ranked_products:
    raise ValueError(
        "The Ranking Agent returned no products."
    )

print(
    f"Ranking completed for {len(ranked_products)} products."
)


Ranking completed for 4 products.


In [26]:
ranking_rows = []

for ranked_product in ranked_products:

    if hasattr(ranked_product, "model_dump"):
        ranked_product_data = ranked_product.model_dump()

    elif isinstance(ranked_product, dict):
        ranked_product_data = ranked_product

    else:
        ranked_product_data = {}

    ranking_rows.append(ranked_product_data)

ranking_df = pd.DataFrame(ranking_rows)

preferred_ranking_columns = [
    "rank",
    "parent_asin",
    "title",
    "retrieval_score",
    "feature_match_score",
    "sentiment_score",
    "rating_score",
    "popularity_score",
    "final_score"
]

display(
    ranking_df[
        [
            column
            for column in preferred_ranking_columns
            if column in ranking_df.columns
        ]
    ]
)


,rank,parent_asin,title,retrieval_score,final_score
0,1,B09KZ9P939,eMorevalue Earbuds Noise Cancelling Magnetic E...,0.451585,0.389634
1,2,B00IMSG0V0,Premium Faux Leather Vertical Swivel Belt Clip...,0.373908,0.284497
2,3,B07D5MRRFG,"Cellet Retractable Stereo in-Ear Headphone, Ea...",0.448896,0.278428
3,4,B094XGTNGN,"Moto G7+ Plus (64GB, 4GB) 6.2"" FHD+ Max Vision...",0.422461,0.250752


In [27]:

for product in products:
    print("\n" + "=" * 80)
    print("Title:", product.get("title"))
    print("Review Summary:", product.get("review_summary"))
    print("Pros:", product.get("pros"))
    print("Recommended For:", product.get("recommended_for"))
    print("Description:", product.get("description"))
    print("Features:", product.get("features"))
    print("Highlights:", product.get("highlights"))
    print("Product Text:", product.get("product_text"))
    print("Categories:", product.get("categories"))


Title: eMorevalue Earbuds Noise Cancelling Magnetic Earphone Headphones for Galaxy A14 5G / A13 5G / A13 / A23 / A12 / A51 / A52 5G / A71 / A03S (A- Black)
Review Summary: No customer reviews available.
Pros: []
Recommended For: 
Description: 
Features: 
Highlights: 
Product Text: Title: eMorevalue Earbuds Noise Cancelling Magnetic Earphone Headphones for Galaxy A14 5G / A13 5G / A13 / A23 / A12 / A51 / A52 5G / A71 / A03S (A- Black)

Description: Fit for Samsung Galaxy A01 A02S A10S A20S A21S A30S A11 A12 A21 A31 A32 A32 5G A42 5G A51 A51 5G A52 A71 A72 A10 A20 A50 A10e A20e, Note 10 Lite, Note 9, S10 Plus, S9 Plus, S8+, S10 5G, S10, S9, S10e, J4 Plus, J8, J2 2018, J2 PureLG V60 V50 V40 V35 G8 G7 ThinQ, Stylo 6, 5, 4, K31 K51S K51 K50 K71 K92 5G K40, Q70 Q51, Reflect, Rebel 4, phoenix 4, G7 Fit Motorola G Play G Power Stylus 2021, G Power Stylus 2020, Moto E, One 5G, G 5G, One Fusion, G Fast, Edge, G9 Play, G8, G8 Power Play Plus, G7 Power Play Plus, E7 Plus, G7, Z4, Z4 Play, G6, G6 

In [28]:
verification = verifier_agent.verify(
    query=query,
    ranking_output=ranking,
    original_products=products
)

print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print(
    "Overall Status:",
    verification.overall_status
)

print(
    "Confidence:",
    verification.confidence
)

print(
    "Recommended Product:",
    verification.recommended_product_title
)

print(
    "Recommended ASIN:",
    verification.recommended_product_asin
)

print()
print("Summary")
print("-" * 40)

print(
    verification.summary
)


FINAL VERIFICATION
Overall Status: failed
Confidence: low
Recommended Product: None
Recommended ASIN: None

Summary
----------------------------------------
Verification finished with status 'failed' and low confidence. All 4 products failed one or more critical checks, so no product could be recommended.


In [29]:
for product in verification.verified_products:

    print("=" * 80)

    print(
        "Rank:",
        product.rank
    )

    print(
        "Title:",
        product.title
    )

    print(
        "Status:",
        product.status
    )

    print(
        "Brand Match:",
        product.brand_match
    )

    print(
        "Product Type Match:",
        product.product_type_match
    )

    print(
        "Feature Status:",
        product.feature_status
    )

    print(
        "Feature Match Ratio:",
        f"{product.feature_match_ratio:.0%}"
    )

    print(
        "Matched Features:",
        product.matched_features
    )

    print(
        "Missing Features:",
        product.missing_features
    )

    print(
        "Budget Verifiable:",
        product.budget_verifiable
    )

    print(
        "Budget Match:",
        product.budget_match
    )

    print(
        "Evidence Strength:",
        product.evidence_strength
    )

    print(
        "Reason:",
        product.verification_reason
    )

    if product.warnings:

        print("\nWarnings")

        for warning in product.warnings:
            print("-", warning)

    print()


Rank: 1
Title: eMorevalue Earbuds Noise Cancelling Magnetic Earphone Headphones for Galaxy A14 5G / A13 5G / A13 / A23 / A12 / A51 / A52 5G / A71 / A03S (A- Black)
Status: failed
Brand Match: False
Product Type Match: True
Feature Status: no_match
Feature Match Ratio: 0%
Matched Features: []
Missing Features: ['good battery life', 'good sound quality']
Budget Verifiable: True
Budget Match: True
Evidence Strength: weak
Reason: Brand match: False. Product type match: True. Feature status: no_match. Feature match ratio: 0.00. Evidence strength: weak. Price available: True. Budget requested: True. Budget match: True.

Warnings
- The product does not clearly match the requested brand.
- None of the requested features could be verified from the available evidence.
- Missing feature evidence: good battery life, good sound quality
- The recommendation has limited review evidence.

Rank: 2
Title: Premium Faux Leather Vertical Swivel Belt Clip Holster for Samsung Galaxy Mega 6.3 Mega 5.8 Nokia 1

In [30]:
verification_df = pd.DataFrame([
    {
        "Rank": product.rank,
        "Parent ASIN": product.parent_asin,
        "Title": product.title,
        "Status": product.status,
        "Brand Match": product.brand_match,
        "Product Type Match": product.product_type_match,
        "Feature Status": product.feature_status,
        "Feature Match Ratio": product.feature_match_ratio,
        "Matched Features": ", ".join(
            product.matched_features
        ),
        "Missing Features": ", ".join(
            product.missing_features
        ),
        "Budget Verifiable": product.budget_verifiable,
        "Budget Match": product.budget_match,
        "Evidence Strength": product.evidence_strength,
        "Warnings": " | ".join(
            product.warnings
        ),
        "Verification Reason": product.verification_reason
    }
    for product in verification.verified_products
])

print("=" * 80)
print("VERIFICATION SUMMARY TABLE")
print("=" * 80)

display(verification_df)


VERIFICATION SUMMARY TABLE


,Rank,Parent ASIN,Title,Status,Brand Match,Product Type Match,Feature Status,Feature Match Ratio,Matched Features,Missing Features,Budget Verifiable,Budget Match,Evidence Strength,Warnings,Verification Reason
0,1,B09KZ9P939,eMorevalue Earbuds Noise Cancelling Magnetic E...,failed,False,True,no_match,0.0,,"good battery life, good sound quality",True,True,weak,The product does not clearly match the request...,Brand match: False. Product type match: True. ...
1,2,B00IMSG0V0,Premium Faux Leather Vertical Swivel Belt Clip...,failed,False,True,no_match,0.0,,"good battery life, good sound quality",True,True,weak,The product does not clearly match the request...,Brand match: False. Product type match: True. ...
2,3,B07D5MRRFG,"Cellet Retractable Stereo in-Ear Headphone, Ea...",failed,False,True,partial_match,0.5,good sound quality,good battery life,False,None,weak,The product does not clearly match the request...,Brand match: False. Product type match: True. ...
3,4,B094XGTNGN,"Moto G7+ Plus (64GB, 4GB) 6.2"" FHD+ Max Vision...",failed,False,True,no_match,0.0,,"good battery life, good sound quality",False,None,weak,The product does not clearly match the request...,Brand match: False. Product type match: True. ...


## Notes

- Restart the Jupyter kernel after changing `Agents/verifier_agent.py` so that the notebook imports the latest class definition.
- The notebook intentionally does not pass a `currency` field because the updated verifier compares only the numeric product price with the numeric budget.
- Missing price results in `passed_with_warning` when all critical checks pass.
- Missing feature evidence results in a warning rather than an automatic failure.
